# Aula 10 - Hiperparâmetros e Explicabilidade do Modelo

**Módulo 03 IN** - Lógica para predição com inteligência artificial
**17/09/2026 - Sprint 4 - Prof. Ovidio Lopes da Cruz Netto**

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/canaldoovidio/2026-2A-M03/blob/main/notebooks/aula10.ipynb)

## O que este notebook é

A Aula 09 terminou com um empate: três modelos marcaram os quatro mesmos valores de acurácia,
precisão, revocação e F1, e nada no material daquela aula conseguia dizer qual deles tinha
aprendido alguma coisa. Este notebook começa desfazendo esse empate com a curva ROC, e depois faz
com a floresta do case o que ninguém tinha feito até aqui: escolher os hiperparâmetros dela em vez
de aceitar os defaults do scikit-learn, validar essa escolha sem olhar para o futuro, e perguntar
ao modelo pronto quais features ele de fato usou.

São quatro medições, e cada uma responde a uma pergunta que a ART.7 faz:

1. **curva ROC e AUC**, que separam três modelos empatados nas outras quatro métricas;
2. **GridSearch e RandomSearch**, que levam a floresta de 5,04% para 4,21% de MAPE;
3. **`TimeSeriesSplit`**, que valida sem treinar com meses posteriores à validação;
4. **SHAP e partial dependence**, que discordam da importância por impureza a partir da terceira
   feature.

Os números batem com `tools/tests/test_ajuste_aula10.py`, e o motivo de cada decisão de escopo
está na `ADR-013`.

## 1. A base analítica mensal e a floresta da Aula 07

A célula abaixo lê as cinco séries mensais de `dados/mensal/` e remonta a base analítica, com a
mesma definição das Aulas 07 a 09, reimplementada aqui porque o notebook precisa rodar sozinho no
Colab:

- junção interna das cinco séries por `periodo`, em ordem cronológica;
- `dias` do mês civil, `sen` e `cos` da posição no ciclo anual;
- `lag1`, `lag2`, `lag3` e `lag12` do abate de frangos, mais as outras quatro séries defasadas em
  um mês;
- descarte das linhas com valor ausente, o que deixa **339 linhas** de 1998-01 a 2026-03;
- corte por data: **315 meses de treino** (1998-01 a 2024-03) e **24 meses de teste** (2024-04 a
  2026-03).

Duas listas de features convivem neste notebook, e a diferença entre elas importa. `FEATURES_07`
são as quatro colunas sobre as quais a Aula 07 treinou a floresta e publicou 4,48% de MAPE.
`FEATURES` são as onze que a Aula 09 deixou. A célula também instala o `shap` quando ele não
estiver presente, o que é o caso no Colab.

In [ ]:
import calendar
import os
import subprocess
import sys
import urllib.request

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import partial_dependence
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import (GridSearchCV, KFold, RandomizedSearchCV,
                                     TimeSeriesSplit)
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

try:
    import shap
except ImportError:
    print("o shap nao esta instalado neste ambiente; instalando (demora cerca de um minuto)")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
    import shap

SERIES = [
    "abate_bovinos",
    "abate_suinos",
    "abate_frangos",
    "producao_ovos",
    "producao_leite",
]
ALVO = "abate_frangos"
FEATURES = (["lag1", "lag2", "lag3", "lag12", "sen", "cos", "dias"]
            + [serie + "_lag1" for serie in SERIES if serie != ALVO])
FEATURES_07 = ["lag1", "lag12", "sen", "cos"]
N_TESTE = 24
SEMENTE = 42

MENSAL_LOCAL = os.path.join("..", "dados", "mensal")
MENSAL_BRUTA = ("https://raw.githubusercontent.com/canaldoovidio/2026-2A-M03/"
                "main/dados/mensal/")

caminhos = {}
for nome in SERIES:
    arquivo = nome + ".csv"
    local = os.path.join(MENSAL_LOCAL, arquivo)
    if os.path.exists(local):
        caminhos[nome] = local
    else:
        if not os.path.exists(arquivo):
            try:
                urllib.request.urlretrieve(MENSAL_BRUTA + arquivo, arquivo)
            except Exception as erro:
                raise RuntimeError(
                    "Nao foi possivel baixar '%s' pela internet (%s). "
                    "Se a rede da sala falhou, peca a pasta 'dados' para uma dupla "
                    "que tenha o repositorio clonado no computador (ela fica na raiz "
                    "do repositorio) e coloque essa pasta ao lado deste notebook. "
                    "Depois, rode esta celula de novo." % (arquivo, erro)
                ) from erro
        caminhos[nome] = arquivo

base = None
for nome in SERIES:
    coluna = (pd.read_csv(caminhos[nome])[["periodo", "valor"]]
              .rename(columns={"valor": nome}))
    base = coluna if base is None else base.merge(coluna, on="periodo", how="inner")
base = base.sort_values("periodo").reset_index(drop=True)

base["mes"] = base["periodo"].str[-2:].astype(int)
base["dias"] = [calendar.monthrange(int(p[:4]), int(p[-2:]))[1] for p in base["periodo"]]
base["sen"] = np.sin(2 * np.pi * base["mes"] / 12)
base["cos"] = np.cos(2 * np.pi * base["mes"] / 12)
for k in (1, 2, 3, 12):
    base["lag%d" % k] = base[ALVO].shift(k)
for nome in SERIES:
    if nome != ALVO:
        base[nome + "_lag1"] = base[nome].shift(1)
base = base.dropna().reset_index(drop=True)

CORTE = len(base) - N_TESTE
X = base[FEATURES].to_numpy(dtype=float)
X07 = base[FEATURES_07].to_numpy(dtype=float)
y = base[ALVO].to_numpy(dtype=float)
lag12 = base["lag12"].to_numpy(dtype=float)
razao = y / lag12

print("linhas: %d   de %s a %s" % (len(base), base["periodo"].iloc[0], base["periodo"].iloc[-1]))
print("treino: %d meses (%s a %s)" % (CORTE, base["periodo"].iloc[0], base["periodo"].iloc[CORTE - 1]))
print("teste:  %d meses (%s a %s)" % (N_TESTE, base["periodo"].iloc[CORTE], base["periodo"].iloc[-1]))
print("features da Aula 09 (%d): %s" % (len(FEATURES), ", ".join(FEATURES)))
print("features da Aula 07 (%d): %s" % (len(FEATURES_07), ", ".join(FEATURES_07)))

## 2. O empate que a Aula 09 deixou

A Aula 09 criou um alvo binário a partir das próprias séries: **o abate do mês supera o mesmo mês
do ano anterior**. O alvo é desbalanceado por natureza, porque o abate de frangos cresceu em 78,1%
dos meses de treino, e a baseline que prevê sempre "cresce" acerta 83,3% dos 24 meses de teste.

A célula abaixo remonta aquele quadro. Repare nas três linhas destacadas: baseline, SVM RBF e
árvore de entropia marcam os quatro mesmos valores, até a terceira casa.

In [ ]:
alvo_binario = (y > lag12).astype(int)
atr, ate = alvo_binario[:CORTE], alvo_binario[CORTE:]

escalador = StandardScaler().fit(X[:CORTE])
Ztr, Zte = escalador.transform(X[:CORTE]), escalador.transform(X[CORTE:])

classificadores = {
    "logistica": LogisticRegression(max_iter=2000, random_state=SEMENTE),
    "naive bayes": GaussianNB(),
    "SVM linear": SVC(kernel="linear", random_state=SEMENTE),
    "SVM RBF": SVC(kernel="rbf", random_state=SEMENTE),
    "arvore entropia": DecisionTreeClassifier(criterion="entropy", max_depth=3,
                                              random_state=SEMENTE),
}

# a baseline preve sempre a classe majoritaria do treino, e o "escore" dela e
# constante: guardar os dois (rotulo e escore) e o que permite medir AUC depois
maioria = np.full(len(ate), int(atr.mean() > 0.5))
previsoes = {"baseline": (maioria, np.zeros(len(ate)))}
for nome, modelo in classificadores.items():
    modelo.fit(Ztr, atr)
    escore = (modelo.decision_function(Zte) if hasattr(modelo, "decision_function")
              else modelo.predict_proba(Zte)[:, 1])
    previsoes[nome] = (modelo.predict(Zte), escore)

print("%-16s %9s %9s %10s %7s" % ("modelo", "acuracia", "precisao", "revocacao", "F1"))
for nome, (previsto, _) in previsoes.items():
    print("%-16s %8.1f%% %9.3f %10.3f %7.3f"
          % (nome, accuracy_score(ate, previsto) * 100,
             precision_score(ate, previsto, zero_division=0),
             recall_score(ate, previsto, zero_division=0),
             f1_score(ate, previsto, zero_division=0)))

Três linhas idênticas. A acurácia, a precisão, a revocação e o F1 dizem que a baseline majoritária,
o SVM RBF e a árvore de entropia são a mesma coisa.

As quatro métricas têm algo em comum: todas são calculadas **depois** de o modelo transformar o que
ele calculou em uma decisão de sim ou não. O SVM calcula uma distância até a fronteira, a árvore
calcula uma proporção de positivos na folha, e as duas coisas viram um rótulo antes de a métrica
ser medida.

A curva ROC não espera essa decisão. Ela varre todos os limiares possíveis e pergunta, em cada um,
quanto se ganha em verdadeiros positivos e quanto se paga em falsos positivos. A AUC resume a curva
num número: a probabilidade de o modelo dar um escore maior a um mês que cresceu do que a um mês
que não cresceu.

In [ ]:
print("%-16s %9s %7s %9s" % ("modelo", "acuracia", "AUC", "pontos ROC"))
for nome, (previsto, escore) in previsoes.items():
    fpr, _, _ = roc_curve(ate, escore)
    print("%-16s %8.1f%% %7.3f %9d"
          % (nome, accuracy_score(ate, previsto) * 100,
             roc_auc_score(ate, escore), len(fpr)))

print()
print("os tres que empatavam:")
for nome in ("baseline", "SVM RBF", "arvore entropia"):
    fpr, tpr, _ = roc_curve(ate, previsoes[nome][1])
    print("  %-16s fpr=%s" % (nome, np.round(fpr, 3).tolist()))
    print("  %-16s tpr=%s" % ("", np.round(tpr, 3).tolist()))

O empate desfez. A baseline vale 0,500, que é o que se espera de quem não ordena nada. A **árvore
de entropia também vale 0,500**, e a curva dela tem dois pontos: ela prevê a mesma classe para os
24 meses, então não existe limiar intermediário para varrer. Em termos de ordenação, a árvore é a
baseline.

O **SVM RBF vale 0,738**. Ele decide igual à baseline no limiar padrão, e por isso empata nas
quatro métricas, mas ordena os meses corretamente com mais frequência do que o acaso. É um modelo
que aprendeu alguma coisa e está com o limiar no lugar errado, o que é um problema com conserto;
a árvore não aprendeu nada, o que não tem conserto por limiar.

Repare também na linha da regressão logística: ela tem a **pior acurácia** entre os modelos que não
desabam (79,2%) e a **melhor AUC** dos cinco (0,800). Ordenar por acurácia e ordenar por AUC dão
listas diferentes, e escolher a métrica é escolher o vencedor.

> **Para a ART.7:** quando dois candidatos empatarem na métrica escolhida, a AUC é a segunda
> pergunta a fazer, antes de decidir por critério subjetivo. E quando o alvo for de regressão, como
> nos três modelos do TAPI, o raciocínio equivalente é comparar contra a baseline declarada, que é
> o protocolo fixado na `ADR-008`.

## 3. Hiperparâmetro não é parâmetro

Um **parâmetro** é o que o modelo aprende a partir dos dados: o coeficiente de cada feature na
regressão linear, o valor de corte de cada nó da árvore. Ninguém escolhe isso à mão; o `fit` é o
ato de calcular.

Um **hiperparâmetro** é o que decide como o `fit` vai acontecer: quantas árvores a floresta tem,
até que profundidade cada uma pode crescer, quantas amostras uma folha precisa ter no mínimo. O
`fit` não escolhe nenhum deles, e o scikit-learn preenche com um default quando ninguém escolhe.

A floresta da Aula 07 tem exatamente um hiperparâmetro escolhido, `n_estimators=300`. Todo o resto
é default. A célula abaixo mostra quais são, e mede a floresta sobre os dois conjuntos de features.

In [ ]:
def mape(real, previsto):
    return float(np.mean(np.abs((real - previsto) / real)) * 100)


def erro_de_teste(modelo, matriz):
    """MAPE de teste em nivel, a partir de um modelo treinado na razao."""
    return mape(y[CORTE:], modelo.predict(matriz[CORTE:]) * lag12[CORTE:])


floresta = RandomForestRegressor(n_estimators=300, random_state=SEMENTE)
print("hiperparametros da floresta da Aula 07:")
for chave in ("n_estimators", "max_depth", "min_samples_leaf", "min_samples_split",
              "max_features"):
    print("  %-20s %s" % (chave, floresta.get_params()[chave]))

print()
f07 = RandomForestRegressor(n_estimators=300, random_state=SEMENTE).fit(X07[:CORTE], razao[:CORTE])
f11 = RandomForestRegressor(n_estimators=300, random_state=SEMENTE).fit(X[:CORTE], razao[:CORTE])
print("a mesma floresta, sem ajuste nenhum:")
print("  sobre as %2d features da Aula 07: MAPE de teste %.2f%%" % (len(FEATURES_07), erro_de_teste(f07, X07)))
print("  sobre as %2d features da Aula 09: MAPE de teste %.2f%%" % (len(FEATURES), erro_de_teste(f11, X)))

As sete features que a Aula 09 acrescentou **pioram** a floresta que ninguém ajustou: 5,04% contra
4,48%. É a maldição de dimensionalidade daquela aula, agora medida no modelo do case e não numa
distância média.

A pergunta do bloco é se isso é culpa das features ou da floresta. Uma floresta com
`max_depth=None` e `min_samples_leaf=1` cresce cada árvore até a folha pura, o que com onze colunas
significa muito mais espaço para decorar ruído do que com quatro. O ajuste de hiperparâmetros é a
forma de descobrir isso sem adivinhar.

O `GridSearchCV` recebe uma grade, treina todas as combinações e devolve a melhor. A grade abaixo
tem 27 combinações, e cada uma é treinada em cinco dobras de validação: 135 treinos, cerca de 9
segundos nesta base de 315 linhas.

In [ ]:
GRADE = {
    "n_estimators": [100, 300, 600],
    "max_depth": [None, 4, 8],
    "min_samples_leaf": [1, 2, 5],
}
combinacoes = len(GRADE["n_estimators"]) * len(GRADE["max_depth"]) * len(GRADE["min_samples_leaf"])

# o MAPE sobre a razao e o mesmo MAPE em nivel, porque o lag12 aparece no
# numerador e no denominador e se cancela: por isso da para buscar direto na
# razao, com o scorer pronto do scikit-learn
busca = GridSearchCV(RandomForestRegressor(random_state=SEMENTE), GRADE,
                     scoring="neg_mean_absolute_percentage_error",
                     cv=TimeSeriesSplit(n_splits=5), n_jobs=-1)
busca.fit(X[:CORTE], razao[:CORTE])

print("%d combinacoes x 5 dobras = %d treinos" % (combinacoes, combinacoes * 5))
print("melhores hiperparametros: %s" % busca.best_params_)
print("MAPE estimado na validacao: %.2f%%" % (-busca.best_score_ * 100))
print("MAPE de teste do escolhido: %.2f%%" % erro_de_teste(busca.best_estimator_, X))
print("MAPE de teste sem ajuste:   %.2f%%" % erro_de_teste(f11, X))

print()
print("saindo do default, um hiperparametro de cada vez:")
partidas = [
    ("default da Aula 07 (None / 1 / 300)", dict(max_depth=None, min_samples_leaf=1, n_estimators=300)),
    ("so podar a profundidade (4 / 1 / 300)", dict(max_depth=4, min_samples_leaf=1, n_estimators=300)),
    ("so exigir folha maior (None / 5 / 300)", dict(max_depth=None, min_samples_leaf=5, n_estimators=300)),
    ("as duas podas juntas (4 / 5 / 300)", dict(max_depth=4, min_samples_leaf=5, n_estimators=300)),
    ("dobrar as arvores, sem podar (None / 1 / 600)", dict(max_depth=None, min_samples_leaf=1, n_estimators=600)),
]
for rotulo, parametros in partidas:
    modelo = RandomForestRegressor(random_state=SEMENTE, **parametros).fit(X[:CORTE], razao[:CORTE])
    print("  %-46s %.2f%%" % (rotulo, erro_de_teste(modelo, X)))

O ajuste leva a floresta de 5,04% para 4,21%, e recupera a maior parte do que as sete features
extras custaram. Não recupera tudo: a floresta ajustada sobre as quatro features da Aula 07 erra
3,95%, e escolher features não é assunto desta aula.

A última tabela mostra de onde vem o ganho. Sair do default podando a profundidade vale cerca de
0,59 ponto percentual, exigir folhas maiores vale 0,64, e fazer as duas coisas juntas vale 0,91.
Dobrar o número de árvores sem podar nada **piora** o erro, de 5,04% para 5,20%. O ganho vem de
podar a árvore, não de ter mais árvores.

Isso tem uma consequência prática: das três dimensões da grade, uma quase não importa, e um terço
dos 135 treinos foi gasto nela. Quando a maior parte da grade não muda o resultado, sortear uma
fração dela encontra a mesma resposta com menos treinos, que é o argumento do
`RandomizedSearchCV`.

In [ ]:
sorteada = RandomizedSearchCV(RandomForestRegressor(random_state=SEMENTE), GRADE,
                              n_iter=9,
                              scoring="neg_mean_absolute_percentage_error",
                              cv=TimeSeriesSplit(n_splits=5),
                              random_state=SEMENTE, n_jobs=-1)
sorteada.fit(X[:CORTE], razao[:CORTE])

print("GridSearch:   %d combinacoes, %d treinos, melhores %s"
      % (combinacoes, combinacoes * 5, busca.best_params_))
print("RandomSearch: %d sorteios,     %d treinos, melhores %s"
      % (9, 9 * 5, sorteada.best_params_))
print()
print("MAPE estimado na validacao: grade %.2f%%   sorteio %.2f%%"
      % (-busca.best_score_ * 100, -sorteada.best_score_ * 100))
print("MAPE de teste:              grade %.2f%%   sorteio %.2f%%"
      % (erro_de_teste(busca.best_estimator_, X), erro_de_teste(sorteada.best_estimator_, X)))

## 4. Validação cruzada temporal

As duas células anteriores usaram `cv=TimeSeriesSplit(n_splits=5)` sem explicação. Esta seção é a
explicação, e ela começa pelo que teria acontecido com o default.

O default do `GridSearchCV` para regressão é `KFold(n_splits=5)`: ele corta os dados em cinco
blocos de tamanho igual, usa um como validação e os outros quatro como treino, cinco vezes. O corte
é por posição na tabela, e a tabela está em ordem de calendário.

A célula abaixo desenha as cinco dobras de cada validador e conta, em cada uma, quantos meses de
treino são **posteriores ao início da validação**.

In [ ]:
periodos = base["periodo"].to_numpy()

for rotulo, cv in (("KFold(5), o default do GridSearchCV", KFold(n_splits=5)),
                   ("TimeSeriesSplit(5)", TimeSeriesSplit(n_splits=5))):
    print(rotulo)
    for i, (itr, ival) in enumerate(cv.split(X[:CORTE]), 1):
        futuro = int((itr > ival.min()).sum())
        print("  dobra %d  treino %3d meses  validacao %3d (%s a %s)  no futuro: %3d"
              % (i, len(itr), len(ival), periodos[ival.min()], periodos[ival.max()], futuro))
    print()

Na primeira dobra do `KFold`, **os 252 meses de treino são todos posteriores ao início da
validação**: o modelo estuda de 2003 a 2024 e é cobrado sobre 1998 a 2003. Quatro das cinco dobras
têm treino no futuro (252, 189, 126, 63 e 0).

O `TimeSeriesSplit` não faz isso em nenhuma dobra. Ele cresce a janela de treino a cada passo, de
55 para 263 meses, e valida sempre no trecho seguinte. É a mesma ideia da repetição em janelas que
a Aula 05 fez à mão, agora como objeto do scikit-learn, e é a generalização do corte único por data
que a Aula 09 usou.

A pergunta honesta é se isso muda o resultado. A célula abaixo mede.

In [ ]:
def escolhido_por(matriz, cv):
    b = GridSearchCV(RandomForestRegressor(random_state=SEMENTE), GRADE,
                     scoring="neg_mean_absolute_percentage_error", cv=cv, n_jobs=-1)
    b.fit(matriz[:CORTE], razao[:CORTE])
    return b.best_params_, -b.best_score_ * 100, erro_de_teste(b.best_estimator_, matriz)


print("%-28s %-46s %12s %10s" % ("base", "hiperparametros escolhidos", "validacao", "teste"))
for nome, matriz in (("onze features (Aula 09)", X), ("quatro features (Aula 07)", X07)):
    for rotulo, cv in (("KFold", KFold(n_splits=5)),
                       ("TimeSeriesSplit", TimeSeriesSplit(n_splits=5))):
        parametros, validacao, teste = escolhido_por(matriz, cv)
        print("%-28s %-46s %11.2f%% %9.2f%%"
              % (nome + ", " + rotulo, str(parametros), validacao, teste))

Quatro linhas que precisam ser lidas com cuidado, porque a conclusão não é a esperada.

**Com as onze features, o `KFold` termina na frente no teste**: 4,13% contra 4,21%. Com as quatro
features, ele termina atrás: 3,99% contra 3,95%. A vantagem troca de sinal conforme o conjunto de
features, e as duas diferenças são menores que um décimo de ponto percentual. Isso não é o
`TimeSeriesSplit` sendo pior; é a diferença sendo **ruído** de um conjunto de teste de 24 meses.

Então por que trocar? Porque o que o `KFold` entrega não é um resultado pior, é uma **estimativa que
não se pode auditar**. Quando ele diz "este modelo erra 6,03% na validação", esse número foi
produzido por um procedimento em que o modelo estudou 2024 para ser cobrado sobre 1998. Nenhum
modelo em produção terá esse privilégio. A estimativa do `TimeSeriesSplit` responde à pergunta que
a LDC vai fazer, que é como o modelo se comporta sobre meses que ele nunca viu e que vieram depois.

É o mesmo raciocínio que a Aula 09 usou para fixar o corte por data: a regressão sobre a razão não
melhorava com o sorteio aleatório, e mesmo assim o corte por data virou regra, porque uma regra de
protocolo vale **antes** de saber qual modelo vai ganhar. Escolher o validador depois de ver quem
ganha é a mesma armadilha que escolher o modelo pela métrica de teste.

## 5. Explicabilidade: o que a floresta usou

O TAPI da LDC pede que o modelo explique as decisões dele, e não só acerte. Uma floresta de 600
árvores podadas não é inspecionável a olho: o que dá para perguntar a ela é quanto cada feature
contribuiu.

Existem duas respostas diferentes para essa pergunta, e elas não concordam.

A **importância por impureza** (`feature_importances_`) vem de graça com a floresta. Ela soma, por
feature, o quanto cada corte que usou aquela feature reduziu o erro dentro do treino, ponderado
pelo número de amostras. É uma leitura sobre o **treino**, e ela tende a inflar features com muitos
valores distintos, que oferecem mais cortes possíveis.

O **SHAP** responde outra coisa: para cada previsão, quanto cada feature empurrou o resultado para
cima ou para baixo em relação à previsão média. É uma leitura por linha, que aqui se mede sobre os
**24 meses de teste**, e é a que responde à pergunta do parceiro, que é sobre previsões que o
modelo vai de fato entregar.

In [ ]:
melhor = busca.best_estimator_
valores_shap = shap.TreeExplainer(melhor).shap_values(X[CORTE:])
medio_shap = np.abs(valores_shap).mean(axis=0)
impureza = melhor.feature_importances_

por_shap = [FEATURES[i] for i in np.argsort(medio_shap)[::-1]]
por_impureza = [FEATURES[i] for i in np.argsort(impureza)[::-1]]

print("%-4s %-22s %-22s" % ("pos", "por SHAP (teste)", "por impureza (treino)"))
for posicao in range(len(FEATURES)):
    marca = " " if por_shap[posicao] == por_impureza[posicao] else "<-"
    print("%-4d %-22s %-22s %s" % (posicao + 1, por_shap[posicao], por_impureza[posicao], marca))

print()
print("a feature mais influente, pelas duas leituras: %s" % por_shap[0])
print("participacao dela no SHAP total: %.1f%%" % (medio_shap[np.argmax(medio_shap)] / medio_shap.sum() * 100))

As duas listas concordam nas duas primeiras posições e **se separam a partir da terceira**.
`abate_bovinos_lag1` é a terceira feature por SHAP e a sexta por impureza; `producao_ovos_lag1` é a
quinta por impureza e a sétima por SHAP.

Não é que uma esteja certa e a outra errada: elas medem coisas diferentes, sobre conjuntos
diferentes. Mas se a dupla relatar "a feature mais influente do nosso modelo" sem dizer qual
leitura usou, a resposta não é verificável, e a ART.7 pede uma resposta verificável.

As duas concordam no que mais importa: `lag12` domina. A floresta prevê a razão entre o mês e o
mesmo mês do ano anterior, então o valor do ano anterior ser a feature dominante já era esperado.
O que não era esperado é **como** ela usa esse valor, e é isso que o partial dependence mostra:
segurando todas as outras features, o que acontece com a previsão quando `lag12` varia.

In [ ]:
topo = int(np.argmax(medio_shap))
resultado = partial_dependence(melhor, X[:CORTE], [topo], grid_resolution=8)
grade_pd = np.asarray(resultado["grid_values"][0], dtype=float)
media_pd = np.asarray(resultado["average"][0], dtype=float)

print("partial dependence de %s, sobre os %d meses de treino" % (FEATURES[topo], CORTE))
print("%-22s %s" % (FEATURES[topo] + " (bilhoes de kg)", "razao prevista"))
for ponto, previsto in zip(grade_pd, media_pd):
    print("%-22.3f %.4f" % (ponto / 1e9, previsto))

print()
print("a curva cai de %.4f para %.4f: %.1f%% de crescimento previsto na base pequena, %.1f%% na base grande"
      % (media_pd[0], media_pd[-1], (media_pd[0] - 1) * 100, (media_pd[-1] - 1) * 100))

A curva é decrescente: quanto maior o abate de doze meses atrás, **menor** o crescimento que a
floresta prevê. Nos meses de base pequena, ela prevê cerca de 9,3% de crescimento sobre o ano
anterior; nos de base grande, cerca de 1,0%.

Isso é lido em uma frase para a LDC, e é uma frase que se pode discutir com quem entende do
negócio: a produção de frango no Brasil cresceu rápido enquanto a base era pequena e desacelerou
conforme ela cresceu, e o modelo aprendeu esse formato sem que ninguém o tenha programado. É
exatamente o tipo de explicação que o parceiro pediu, e é o oposto de "a floresta tem 600 árvores".

## O que levar para a ART.7

**ART.7 Comparação de modelos, peso 8**, é a entrega da Sprint 4, com planning em 14/09 e review em
25/09. O que esta aula acrescenta ao protocolo que a Aula 09 fixou:

1. **hiperparâmetro escolhido, e não herdado**. Todo candidato da comparação precisa dizer quais
   hiperparâmetros foram buscados, em que grade, e sob qual validação. Aceitar o default é uma
   escolha, e ela precisa estar declarada como tal;
2. **validação cruzada que respeita o tempo** (`TimeSeriesSplit`), pelo mesmo motivo que o corte por
   data virou regra na Aula 09: o protocolo se decide antes de saber quem ganha;
3. **métrica de desempate** quando dois candidatos empatarem: AUC no alvo categórico, baseline
   declarada no alvo de regressão;
4. **importância de feature com a leitura declarada**: SHAP sobre o teste e impureza sobre o treino
   dão listas diferentes a partir da terceira posição, então dizer qual foi usada faz parte da
   resposta.

O modelo que sai daqui, floresta com `max_depth=4`, `min_samples_leaf=5` e `n_estimators=600` a
4,21% de MAPE, é o candidato ajustado à mão. Na Aula 11 ele vira o termo de comparação contra o
PyCaret: a pergunta de abertura daquela aula é se um comparador automático bate o que a dupla
ajustou sabendo o que estava fazendo.